In [18]:
import pandas as pd
from pathlib import Path

q1 = pd.read_csv("../data/raw/Q1_2021.csv")
q2 = pd.read_csv("../data/raw/Q2_2021.csv")
directorio = pd.read_csv("../data/raw/Directorio.csv", encoding="latin-1")

In [19]:
directorio = directorio.drop(columns=["Unnamed: 2"], errors="ignore")
print("Directorio:", directorio.shape)
print(directorio.head(10))

Directorio: (30, 4)
   Client  Branch      Nombre        Estado
0    1520       1  Sucursal A       Yucatán
1    1520       2  Sucursal B       Yucatán
2    1520       3  Sucursal C       Yucatán
3    1520       5  Sucursal D       Yucatán
4    1521       6  Sucursal E       Yucatán
5    1521       7  Sucursal F  Quintana Roo
6    1521       8  Sucursal G  Quintana Roo
7    1521       9  Sucursal H  Quintana Roo
8    1521      10  Sucursal I  Quintana Roo
9    1521      11  Sucursal J  Quintana Roo


In [20]:
columnas_fecha = ["FechaFactura", "FechaApertura", "FechaEntrega"]

columnas_moneda = [
    "Venta$", "Costo$", "Utilidad$",
    "VentaMO$", "DescuentoMO$", "CostoMO$",
    "VentaMateriales$", "DescuentoMateriales$", "CostoMateriales$",
    "VentaTOT$", "DescuentoTOT$", "CostoTOT$",
    "VentaPartes$", "DescuentoPartes$", "CostoPartes$",
    "VentaTotal$", "CostoTotal$",
]

columnas_entero = [
    "IdRegistro", "Client", "Branch", "Factura",
    "NumeroOT", "VentasNetas", "Odometro", "Ano",
]

columnas_decimal = ["Margen", "Dias"]


def homologar_columnas(df):
    df = df.copy()

    # Fechas: formato de día/mes/año
    for columna in columnas_fecha:
        if columna in df.columns:
            df[columna] = pd.to_datetime(
                df[columna],
                dayfirst=True,
                errors="coerce"
            )

    
    for columna in columnas_moneda:
        if columna in df.columns:
            valores = (
                df[columna]
                .astype("string")
                .str.replace(r"[$,\s]", "", regex=True)
            )
            df[columna] = pd.to_numeric(valores, errors="coerce")


    for columna in columnas_entero:
        if columna in df.columns:
            df[columna] = pd.to_numeric(
                df[columna],
                errors="coerce"
            ).astype("Int64")

 
    for columna in columnas_decimal:
        if columna in df.columns:
            df[columna] = pd.to_numeric(
                df[columna],
                errors="coerce"
            ).astype("Float64")

   
    columnas_convertidas = (
        columnas_fecha
        + columnas_moneda
        + columnas_entero
        + columnas_decimal
    )
    for columna in df.columns:
        if columna not in columnas_convertidas:
            df[columna] = df[columna].astype("string")

    return df



q1 = homologar_columnas(q1)
q2 = homologar_columnas(q2)


In [21]:
q1["Fuente"] = "Q1"
q2["Fuente"] = "Q2"


datos = pd.concat([q1, q2], ignore_index=True)


columnas_vacias = datos.columns[datos.isna().all()]
datos = datos.drop(columns=columnas_vacias)

print("Columnas eliminadas por estar completamente vacías:", list(columnas_vacias))
print("Tamaño del conjunto unido:", datos.shape)
print("\nTipos de datos:")
print(datos.dtypes)

Columnas eliminadas por estar completamente vacías: ['Modelo']
Tamaño del conjunto unido: (120258, 37)

Tipos de datos:
FechaFactura            datetime64[us]
IdRegistro                       Int64
Client                           Int64
Branch                           Int64
FechaApertura           datetime64[us]
FechaEntrega            datetime64[us]
Factura                          Int64
Taller                          string
TipoOrden                       string
TipoPago                        string
NumeroOT                         Int64
Venta$                         Float64
Costo$                         Float64
Utilidad$                      Float64
Margen                         Float64
VentasNetas                      Int64
VentaMO$                       Float64
DescuentoMO$                   Float64
CostoMO$                       Float64
VentaMateriales$               Float64
DescuentoMateriales$           Float64
CostoMateriales$               Float64
VentaTOT$             

In [22]:
referencia = directorio[
    ["Client", "Branch", "Nombre", "Estado"]
].drop_duplicates()

datos = datos.merge(
    referencia,
    on=["Client", "Branch"],
    how="left",
    validate="many_to_one"
)

datos[["Nombre", "Estado"]] = datos[["Nombre", "Estado"]].fillna("Sin Definir")

In [25]:
conteo_sin_definir = datos[["Nombre", "Estado"]].eq("Sin Definir").sum()
print(conteo_sin_definir)

Nombre    0
Estado    0
dtype: int64


In [26]:
datos[["Fuente", "IdRegistro", "Client", "Branch","Nombre","Estado"]].head(10)

,Fuente,IdRegistro,Client,Branch,Nombre,Estado
0,Q1,995439,1522,20,Sucursal O,Tabasco
1,Q1,995440,1522,20,Sucursal O,Tabasco
2,Q1,571394,1521,11,Sucursal J,Quintana Roo
3,Q1,995441,1522,20,Sucursal O,Tabasco
4,Q1,995442,1522,20,Sucursal O,Tabasco
5,Q1,995443,1522,20,Sucursal O,Tabasco
6,Q1,995444,1522,20,Sucursal O,Tabasco
7,Q1,995445,1522,20,Sucursal O,Tabasco
8,Q1,995446,1522,20,Sucursal O,Tabasco
9,Q1,995493,1522,20,Sucursal O,Tabasco


In [27]:
carpeta_salida = Path("../data/preprocessed")
carpeta_salida.mkdir(parents=True, exist_ok=True)

ruta_salida = carpeta_salida / "datos_2021.csv"
datos.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print(f"Archivo guardado en: {ruta_salida}")

Archivo guardado en: ..\data\preprocessed\datos_2021.csv
